# Restricted-input method reference
Source: analysis/intra-park/proximity_calculation/park_service.ipynb
Requires independently obtained inputs and source directory layout. Not an end-to-end pipeline. Outputs can contain sensitive data; keep the working directory private. External absolute paths have been replaced by external_inputs/<basename>. See docs/REPRODUCIBILITY.md.


In [ ]:
import os
from pathlib import Path
private_root = Path(os.environ["UFUG_PRIVATE_ROOT"]).resolve()
os.chdir(private_root)


In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

EPSG_SG = 3414  # SVY21 / Singapore TM, metres

import os
# Working directory is explicitly configured in the first cell.


In [ ]:
def _read_points(path) -> gpd.GeoDataFrame:
    """Read points from CSV (lon,lat in WGS84) or Point GeoJSON; return in EPSG:3414."""
    p = str(path).lower()
    if p.endswith((".geojson", ".json", ".gpkg", ".shp")):
        gdf = gpd.read_file(path)
        if gdf.crs is None:
            gdf = gdf.set_crs(4326)  # assume WGS84 if missing
    else:
        df = pd.read_csv(path)
        if not {"lon", "lat"}.issubset(df.columns):
            raise ValueError("CSV must contain 'lon' and 'lat' columns.")
        gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326)
    return gdf.to_crs(EPSG_SG)

def count_pois_within_radius(points_path, pois_path, radius_m=500, poi_id_col=None):
    """
    Count POIs within `radius_m` of each point.

    Parameters
    ----------
    points_path : str/Path
        CSV (lon,lat) or Point GeoJSON for the target points.
    pois_path : str/Path
        CSV (lon,lat) or Point GeoJSON for POIs.
    radius_m : float
        Search radius in metres (EPSG:3414 units).
    poi_id_col : str or None
        Optional column in POIs to aggregate as a list per target (e.g., 'poi_id').

    Returns
    -------
    GeoDataFrame of target points with:
      - count_within_{radius_m}m
      - (optional) poi_ids_within_{radius_m}m  (comma-separated)
    """
    pts = _read_points(points_path)
    pois = _read_points(pois_path)

    if pois.empty:
        raise ValueError("POI layer has no features.")

    # Build 500 m buffers around each target point
    buffers = pts.copy()
    buffers["geometry"] = buffers.geometry.buffer(radius_m)

    # Spatial join: which POIs fall in each buffer
    # predicate='intersects' is correct because POIs are inside the buffer polygons
    joined = gpd.sjoin(buffers[["geometry"]], pois[["geometry"]], how="left", predicate="intersects")

    # Count POIs per target (index_left refers to the target point index)
    counts = joined.groupby(joined.index).size().rename(f"count_within_{radius_m}m")
    counts = counts.reindex(pts.index, fill_value=0)

    # Optional: collect POI IDs as a comma-separated list
    if poi_id_col and poi_id_col in pois.columns:
        joined_ids = gpd.sjoin(
            buffers[["geometry"]], pois[[poi_id_col, "geometry"]], how="left", predicate="intersects"
        )
        id_series = (
            joined_ids.groupby(joined_ids.index)[poi_id_col]
            .apply(lambda s: ",".join(map(str, s.dropna().unique())))
            .rename(f"poi_ids_within_{radius_m}m")
        )
        id_series = id_series.reindex(pts.index, fill_value="")
        out = pts.copy()
        out[counts.name] = counts.values
        out[id_series.name] = id_series.values
    else:
        out = pts.copy()
        out[counts.name] = counts.values

    return out

In [ ]:
for park in ["jurong", "amk", "punggol"]:
    gdf_base = gpd.read_file(f'gis_files/hexagons/nearest_mrt/parks_nearest_mrt_{park}.geojson')
    targets_path = f"analysis/path_calculation/origins_{park}_coords.csv"      # CSV with lon,lat OR a Point GeoJSON
    pois_path    = f"gis_files/poi/poi_{park}_clipped.geojson"     # CSV with lon,lat OR a Point GeoJSON
    result = count_pois_within_radius(targets_path, pois_path, radius_m=500, poi_id_col="poi_id")
    result.drop(columns="geometry").head()
    result = result[['grid_id', 'count_within_500m']]
    result.to_csv(f"analysis/intra-park/proximity_calculation/targets_with_poi_counts_{park}.csv", index=False)  # or .to_file(...)
    gdf_new = gdf_base.merge(result, on='grid_id', how='left')
    mean_count = gdf_new['count_within_500m'].mean()
    median_count = gdf_new['count_within_500m'].median()
    std_count = gdf_new['count_within_500m'].std()
    print(f"For {park} park: Mean POI count within 500m: {mean_count}, Median: {median_count}, Std Dev: {std_count}")
    gdf_new.to_file(f"analysis/intra-park/proximity_calculation/parks_nearest_mrt_{park}_with_poi_counts.geojson", driver="GeoJSON")

In [ ]:
gdf_new